In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Note on Load. Refer to the archived source for the detailed original comment.
classes_path = "/kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/labels/classes.txt"
with open(classes_path, 'r') as f:
    class_names = [line.strip() for line in f.readlines()]

nc = len(class_names)
names = {i: name for i, name in enumerate(class_names)}

print(f"检测到 {nc} 个类别:")
for i, name in names.items():
    print(f"  {i}: {name}")

# 写入 d2-city.yaml
yaml_content = f"""
# D2-City dataset configuration
path: /kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset
train: /kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/images/train
val: //kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/images/val

nc: {nc}
names: {names}
"""

with open("/kaggle/working/d2-city.yaml", "w") as f:
    f.write(yaml_content)

print("已生成 /kaggle/working/d2-city.yaml")

检测到 12 个类别:
  0: car
  1: van
  2: bus
  3: truck
  4: person
  5: bicycle
  6: motorcycle
  7: open-tricycle
  8: closed-tricycle
  9: forklift
  10: large-block
  11: small-block
已生成 /kaggle/working/d2-city.yaml


In [3]:
%%writefile train_exp.py
# train_exp.py 

import os
import sys
from ultralytics import YOLO

"""
Part II: YOLO26s - 更换为更大模型并加载 COCO 预训练权重
"""

def experiment_1():
    """实验1: YOLO26s + pretrained (640px)"""
    model = YOLO("yolo26s.pt")          # 加载预训练权重
    
    model.train(
        data="./d2-city.yaml",
        epochs=100,
        cache=True,        
        imgsz=800,
        batch=16,          # 在显存允许下batch 越大，GPU 利用率越高。对于 26s，若开启 AMP，batch 可尝试 16。
        amp=True,
        rect=True,         # 矩形训练减少填充，提高训练效率
        workers=8,                 # 增加数据加载线程
        freeze=10,                 # 冻结前 10 层，加快训练（可选）
        save_period=10,            # 每 10 轮保存一个检查点
        project=".",            # 保存根目录
        name="exp1"             # 子目录
    )
    
    metrics = model.val()
    print(f"Exp1 mAP50-95: {metrics.box.map:.4f}")
    print(f"Exp1 mAP50: {metrics.box.map50:.4f}")

"""
Part III: YOLO26n + 预训练权重 + 更高分辨率且数据增强
"""

def experiment_2():
    """实验2: YOLO26n + pretrained + 高分辨率960px + 数据增强"""
    model = YOLO("yolo26n.pt")
    
    model.train(
        data="./d2-city.yaml",
        epochs=100,
        cache=False,        # Kaggle 关闭缓存
        imgsz=960,
        batch=10,
        rect=True,          # 矩形训练，减少填充
        mosaic=1.0,
        mixup=0.5,
        copy_paste=0.5,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.4,
        amp=False,
        project=".",
        name="exp2"
    )
    metrics = model.val()
    print(f"Exp2 mAP50-95: {metrics.box.map:.4f}")
    print(f"Exp2 mAP50: {metrics.box.map50:.4f}")

"""
Part IV: YOLO26m + 预训练权重 + 更高分辨率且数据增强、优化器等
"""

def experiment_3():
    """
    实验3: 激进方案 —— 更大模型 + 高分辨率 + 全面优化
    - 模型: YOLO26m (参数量 ~25M)
    - 分辨率: 960px
    - 优化器: AdamW + 余弦退火
    - 训练轮数: 150, 早停 patience=50
    - 增强: mosaic, mixup, copy_paste, flipud, degrees, scale
    - 标签平滑: 0.1
    - 验证时启用 TTA
    """
    model = YOLO("yolo26m.pt")          # 最大的 YOLO26 模型
    
    model.train(
        data="./d2-city.yaml",
        cache=True,
        epochs=150,
        patience=50,
        imgsz=800,           # 降低至 800，平衡精度与速度
        batch=8,             # 开启AMP
        rect=True,
        optimizer='AdamW',
        lr0=0.001,
        cos_lr=True,
        label_smoothing=0.1,
        mosaic=0.8,
        mixup=0.4,
        copy_paste=0.5,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.4,
        flipud=0.5,
        degrees=5.0,
        scale=0.5,
        amp=True,               # 若显存不足可开启 amp=True 节省显存
        project=".",
        name="exp3"
    )
    
    # Note on Validation. Refer to the archived source for the detailed original comment.
    metrics = model.val(augment=True)
    print(f"Exp3 mAP50-95: {metrics.box.map:.4f}")
    print(f"Exp3 mAP50: {metrics.box.map50:.4f}")

"""
Part V: YOLOv11n - 速度与效率对照实验
"""

def experiment_4():
    """实验4: YOLOv11n - 速度与效率对照实验"""
    model = YOLO("yolo11n.pt")  # 或 yolo11s.pt
    
    model.train(
        data="./d2-city.yaml",
        epochs=150,             # 与实验2/3保持一致
        patience=50,
        cache=True,
        imgsz=800,              # 从标准分辨率开始
        batch=16,
        amp=True,               # 启用混合精度训练
        rect=True,
        project=".",
        name="exp4"
    )
    
    metrics = model.val()
    print(f"Exp4 (YOLOv11n) mAP50-95: {metrics.box.map:.4f}")
    print(f"Exp4 (YOLOv11n) mAP50: {metrics.box.map50:.4f}")

"""
Part VI: RT-DETR - Transformer 架构实验
"""

def experiment_5():
    """实验5: RT-DETR-L 在 D2-City 数据集上的表现"""
    from ultralytics import RTDETR

    # Note on Model, Performance, Use. Refer to the archived source for the detailed original comment.
    model = RTDETR("rtdetr-l.pt")

    model.train(
        data="./d2-city.yaml",
        epochs=1,
        #patience=50,
        imgsz=640,          # RT-DETR 默认推荐 640
        batch=8,            # Transformer 模型通常更耗显存，建议从较小的 batch size 开始
        amp=True,
        project=".",
        name="exp5"
    )

    metrics = model.val()
    print(f"Exp5 (RT-DETR-L) mAP50-95: {metrics.box.map:.4f}")
    print(f"Exp5 (RT-DETR-L) mAP50: {metrics.box.map50:.4f}")

if __name__ == "__main__":
    # Note on Experiment, Parameters. Refer to the archived source for the detailed original comment.
    # Note on Experiment, Run. Refer to the archived source for the detailed original comment.
    if len(sys.argv) > 1:
        if sys.argv[1] == "2":
            experiment_2()
        elif sys.argv[1] == "3":
            experiment_3()
        elif sys.argv[1] == "4":
            experiment_4()
        elif sys.argv[1] == "5":
            experiment_5()
        else:
            print(f"未知参数: {sys.argv[1]}, 使用默认实验1")
            experiment_1()
    else:
        experiment_1()

Writing train_exp.py


In [4]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 660.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.7 MB/s eta 0:00:00


In [5]:
# Note on Experiment, Run. Refer to the archived source for the detailed original comment.
!python train_exp.py

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./d2-city.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscri

In [6]:
from ultralytics import YOLO
import os
import random

# Note on Model, Training, Load. Refer to the archived source for the detailed original comment.
model = YOLO("/kaggle/working/runs/detect/exp1/weights/best.pt")  # 或 last.pt

# Note on Testing, Image. Refer to the archived source for the detailed original comment.
test_img_dir = "/kaggle/input/datasets/xiaoyuwan/didi-detection/dataset/dataset_release/test"

# 随机选8个编号 (11000~11999)
indices = random.sample(range(11000, 12000), 8)
test_img_files = [os.path.join(test_img_dir, f"{idx}.jpg") for idx in indices]

# Note on Prediction. Refer to the archived source for the detailed original comment.
results = model(test_img_files)

# Note on Results. Refer to the archived source for the detailed original comment.
for file, result in zip(test_img_files, results):
    print(f"预测图像: {os.path.basename(file)}")
    result.show()


0: 800x800 6 cars, 1 van, 19.3ms
1: 800x800 4 cars, 2 vans, 19.3ms
2: 800x800 8 cars, 2 buss, 19.3ms
3: 800x800 4 cars, 1 small-block, 19.3ms
4: 800x800 8 cars, 1 van, 1 bus, 19.3ms
5: 800x800 4 cars, 1 bus, 1 person, 19.3ms
6: 800x800 4 cars, 19.3ms
7: 800x800 13 cars, 1 bus, 1 person, 19.3ms
Speed: 5.2ms preprocess, 19.3ms inference, 0.3ms postprocess per image at shape (1, 3, 800, 800)
预测图像: 11047.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

预测图像: 11596.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

预测图像: 11511.jpg


<PIL.Image.Image image mode=RGB size=1280x720>

... (6 outputs left out)

预测图像: 11771.jpg


<PIL.Image.Image image mode=RGB size=1280x720>

预测图像: 11001.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>